# Notebook 08 — Frontend Data Export

**AI Evolution Map · Export Pipeline**

---

## 1. Export Objective

This notebook produces validated, schema-compliant JSON and CSV artifacts formatted specifically for the three target visual interfaces:

1. **Orbit Timeline**: Chronological milestones with era categorization, confidence rating, date precision, and cited source IDs.
2. **AI Family Tree**: Directed graph of concepts, categories, and evidence-backed parent-child/successor relationships.
3. **REAVIZ Analytics**: Quantitative metric observations and aggregate research distributions.

**Golden Principle**: The React frontend must NEVER perform data cleaning or schema repairs. Every dataset exported here is pre-validated, timestamped, versioned, and structurally final.

In [1]:
import sys
from pathlib import Path

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
from datetime import datetime, timezone
import pandas as pd
from src.export.exporters import export_to_json, export_to_csv, export_to_parquet
from src.common.schemas import Milestone, Concept, Relationship, ResearchSource, MetricObservation

PROCESSED_DIR = project_root / 'data' / 'processed'
REFERENCE_DIR = project_root / 'data' / 'reference'
EXPORT_DIR = project_root / 'data' / 'processed' / 'frontend_export'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Export destination: {EXPORT_DIR}')

Export destination: C:\Users\MANAS\Documents\AI-Evolution-Map\data\processed\frontend_export


## 2. Ingest Reference Knowledge Base (Milestones, Concepts, Relationships)

We load canonical verified entities from `data/reference/seed_knowledge_base.json`.
Every entity is passed through its corresponding Pydantic schema before export.

In [2]:
kb_path = REFERENCE_DIR / 'seed_knowledge_base.json'
if not kb_path.exists():
    print(f'Seed knowledge base not found at: {kb_path}')
    seed_data = {'milestones': [], 'concepts': [], 'relationships': [], 'sources': [], 'metrics': []}
else:
    with open(kb_path, encoding='utf-8') as f:
        seed_data = json.load(f)

print(f'Loaded seed entities: '
      f'{len(seed_data.get("milestones", []))} milestones, '
      f'{len(seed_data.get("concepts", []))} concepts, '
      f'{len(seed_data.get("relationships", []))} relationships, '
      f'{len(seed_data.get("sources", []))} sources, '
      f'{len(seed_data.get("metrics", []))} metrics.')

Loaded seed entities: 8 milestones, 10 concepts, 7 relationships, 10 sources, 3 metrics.


## 3. Strict Pydantic Validation of Export Payloads

In [3]:
# Validate seed knowledge base entities with Pydantic
validated_milestones = [Milestone(**m).model_dump() for m in seed_data.get('milestones', [])]
validated_concepts = [Concept(**c).model_dump() for c in seed_data.get('concepts', [])]
validated_relationships = [Relationship(**r).model_dump() for r in seed_data.get('relationships', [])]
validated_sources = [ResearchSource(**s).model_dump() for s in seed_data.get('sources', [])]
validated_seed_metrics = [MetricObservation(**met).model_dump() for met in seed_data.get('metrics', [])]

# Load pre-validated metrics from data/processed/ (44 multi-year + 3 seed = 47 total)
metrics_val_path = PROCESSED_DIR / 'metrics_validated.json'
if metrics_val_path.exists():
    with open(metrics_val_path, encoding='utf-8') as f:
        all_validated_metrics = [MetricObservation(**m).model_dump() for m in json.load(f)]
    print(f'Loaded {len(all_validated_metrics)} validated metric records from data/processed/.')
else:
    all_validated_metrics = validated_seed_metrics

print(f'Total validated metrics to export: {len(all_validated_metrics)}')
print('All entities validated successfully against Pydantic contract!')


Loaded 47 validated metric records from data/processed/.
Total validated metrics to export: 47
All entities validated successfully against Pydantic contract!


## 4. Exporting Frontend Artifacts

1. `orbit_timeline.json`: For chronological visualization
2. `concept_graph.json`: Nodes (concepts) and links (relationships) for graph/family tree
3. `research_sources.json`: Provenance citations
4. `analytics_metrics.json` and `.csv`: Quantitative trends

In [4]:
# 1. Orbit Timeline Export
export_to_json(validated_milestones, EXPORT_DIR / 'orbit_timeline.json', 'milestone', 'Timeline chronological events')
export_to_csv(validated_milestones, EXPORT_DIR / 'orbit_timeline.csv', 'milestone')

# 2. Concept Graph (Family Tree) Export
graph_payload = {
    'nodes': validated_concepts,
    'edges': validated_relationships
}
export_to_json([graph_payload], EXPORT_DIR / 'concept_graph.json', 'concept_graph', 'Nodes and directed edges')

# 3. Sources Export
export_to_json(validated_sources, EXPORT_DIR / 'research_sources.json', 'research_source', 'Bibliographic citations')
export_to_csv(validated_sources, EXPORT_DIR / 'research_sources.csv', 'research_source')

# 4. Analytics Metrics Export (Multi-Year Series)
export_to_json(all_validated_metrics, EXPORT_DIR / 'analytics_metrics.json', 'metric_observation', 'Quantitative observations')
export_to_csv(all_validated_metrics, EXPORT_DIR / 'analytics_metrics.csv', 'metric_observation')
export_to_parquet(all_validated_metrics, EXPORT_DIR / 'analytics_metrics.parquet', 'metric_observation')

print('Frontend exports successfully written to:', EXPORT_DIR)


Frontend exports successfully written to: C:\Users\MANAS\Documents\AI-Evolution-Map\data\processed\frontend_export
